# Surrogate model persistence

Compiling a surrogate model is the expensive one-time step; the previous notebook
showed that evaluating it afterwards is a cheap classical calculation. This notebook
covers the other half of that lifecycle: saving a compiled model to disk with
`save`/`load` so it doesn't need to be rebuilt in a later session, the `n_terms`/
`n_monomials` diagnostics for gauging how much computation a compiled model actually
represents, and `evaluate_batch` for evaluating many parameter assignments at once
instead of looping over `evaluate`.

## Building the same ansatz as before

We'll reuse the exact small ansatz and observable from the previous notebook, so we
have a `PauliSurrogateModel` to work with without repeating the optimization loop.

In [1]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import XXPlusYYGate
from qiskit.quantum_info import SparsePauliOp

n_qubits = 3
reps = 2

theta = ParameterVector("theta", n_qubits * reps)
phi = ParameterVector("phi", (n_qubits - 1) * reps)

qc = QuantumCircuit(n_qubits)

# HF State
qc.x(0)
qc.x(2)

it, ip = 0, 0
for layer in range(reps):
    for q in range(n_qubits):
        qc.rz(theta[it], q)
        it += 1
    for q in range(n_qubits - 1):
        qc.append(XXPlusYYGate(phi[ip], 0.0), [q, q + 1])
        ip += 1

observable = SparsePauliOp.from_list([("ZZI", 1.0), ("IZZ", 1.0), ("ZIZ", 0.5)])

In [2]:
from propaq.circuits import SurrogatePauliCircuit
from propaq.datatypes import PauliTermSum
from propaq.propagators import PauliSurrogatePropagator
from propaq.models import VariationalSurrogateModel

obs_term_sum = PauliTermSum.from_sparse_pauli_op(observable)
surrogate_circuit = SurrogatePauliCircuit.from_qiskit(qc)

model = PauliSurrogatePropagator().build(obs_term_sum, surrogate_circuit, initial_state=0)
variational_model = VariationalSurrogateModel(
    model, surrogate_circuit.parameter_sources, surrogate_circuit.qiskit_parameters
)

## Saving and loading a compiled model

`save`/`load` round-trip a model through a gzip-compressed binary file. Loading
skips propagation entirely: the compiled model is ready to `evaluate` immediately,
so a later session (or a different machine) doesn't need the original circuit or
observable at all, only the file.

In [3]:
from propaq._rust_core import PauliSurrogateModel

model.save("surrogate_model.pq")
loaded_model = PauliSurrogateModel.load("surrogate_model.pq")

rng = np.random.default_rng(0)
params = rng.uniform(-np.pi, np.pi, size=model.n_params).tolist()

print("Original model evaluate: ", model.evaluate(params))
print("Loaded model evaluate:   ", loaded_model.evaluate(params))
print("Match:", model.evaluate(params) == loaded_model.evaluate(params))

Original model evaluate:  -0.8858230090358632
Loaded model evaluate:    -0.8858230090358632
Match: True


## Model diagnostics: `n_terms` and `n_monomials`

`n_terms` is the number of compiled terms surviving in the model (zero-overlap terms
with the initial state are excluded, since they can never contribute to any
expectation value). `n_monomials` is a different, usually much larger number: the
total pre-dedup monomial-instance count across every surviving term's coefficient.
A model can look small by `n_terms` alone while still representing an enormous
amount of underlying symbolic computation, which is what `n_monomials` actually tracks.

In [4]:
print("n_params:   ", model.n_params)
print("n_terms:    ", model.n_terms)
print("n_monomials:", model.n_monomials)

n_params:    10
n_terms:     3
n_monomials: 159


## Batched evaluation

`evaluate_batch` evaluates many parameter assignments in one call, parallelized
across the assignments, instead of calling `evaluate` in a Python loop. This is the
natural fit for anything that needs the cost function at many points at once --
scanning a parameter, an ensemble of optimizer restarts, or (as here) just a batch
of random parameter sets.

In [5]:
import time

n_sets = 500
param_sets = [rng.uniform(-np.pi, np.pi, size=model.n_params).tolist() for _ in range(n_sets)]

t0 = time.perf_counter()
looped = [model.evaluate(p) for p in param_sets]
t1 = time.perf_counter()
batched = model.evaluate_batch(param_sets)
t2 = time.perf_counter()

print(f"Looped evaluate:  {(t1 - t0) * 1000:.2f} ms for {n_sets} parameter sets")
print(f"evaluate_batch:   {(t2 - t1) * 1000:.2f} ms for {n_sets} parameter sets")
print("Results match (within floating-point tolerance):", np.allclose(looped, batched))
print("Largest discrepancy:", max(abs(a - b) for a, b in zip(looped, batched)))

Looped evaluate:  20.15 ms for 500 parameter sets
evaluate_batch:   0.95 ms for 500 parameter sets
Results match (within floating-point tolerance): True
Largest discrepancy: 2.220446049250313e-16
